In [3]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from pydantic import BaseModel, Field
from typing import List
import os
from pydantic import BaseModel
from typing import List
from crewai_tools import JSONSearchTool
import numpy as np
import chromadb
from openai import OpenAI
import json
import faiss

In [4]:
openai_api_key = os.environ.get("OPENAI_API_KEY")

In [8]:
output_dir = "./ai-agent-chatbot-output"
os.makedirs(output_dir, exist_ok=True)

basic_llm = LLM(model="openai/gpt-4o", temperature=0)

In [5]:
with open('/media/ahmed/New Volume/crocoit/learn/raw_pages.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [9]:
chunks = []
for item in data:
    chunks.append({
        "text": f"{item['title']}\n{item['content']}",
        "url": item["url"],
        "title": item["title"]
    })

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def get_embedding(text: str):
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="crocoit")

collection.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=[get_embedding(c["text"]) for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"title": c["title"], "url": c["url"]} for c in chunks]
)

In [10]:
class CrocoitSource(BaseModel):
    title: str
    url: str

class CrocoitOutput(BaseModel):
    response: str
    sources: List[CrocoitSource]



@tool
def json_search_tool(query: str) -> list:
    """Use this tool ONLY when the user asks about CrocoIT, its services, products, solutions, or any information related to the company CrocoIT. Do NOT use this tool for general questions unrelated to CrocoIT."""
    
    query_vec = get_embedding(query)
    
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=3
    )
    
    output = []
    for i in range(len(results["documents"][0])):
        output.append({
            "title": results["metadatas"][0][i]["title"],
            "content": results["documents"][0][i],
            "url": results["metadatas"][0][i]["url"],
        })
    
    return output if output else "No relevant information found."



crocoit_chatbot_agent =Agent(
    role="Crocoit Technology Solutions Chatbot",
    goal="\n".join([
    "Answer based on information retrieved from the JSON knowledge base tool if the user asks about CrocoIT.",
    "Use your own knowledge if the user does not ask about CrocoIT.",
    ]),
    backstory="An intelligent agentic chatbot built for Crocoit, a technology solutions company. The agent reads and processes structured data from a JSON knowledge base to provide users with precise, context-aware responses about Crocoit's services and offerings.",
    llm=basic_llm,
    verbose=True,
    tools=[json_search_tool])


crocoit_chatbot_task =Task(
    description="\n".join([
        "Answer the user query: {user_query}",
        "If the query is about CrocoIT, use json_search_tool to retrieve information.",
        "If the query is NOT about CrocoIT, answer from your own knowledge.",
        "Do NOT use prior knowledge for CrocoIT-related questions.",
        "when query related to crocoit send sumry and direct answer not the source text"
    ]),
    expected_output="A JSON object with 'response' containing a clear answer, and 'sources' containing a list of title and url for each source used.",
    output_json=CrocoitOutput,
    output_file=os.path.join(output_dir, "step_3_search_results.json"),
    agent=crocoit_chatbot_agent
)


In [11]:
rankyx_crew = Crew(
    agents=[
        crocoit_chatbot_agent
    ],
    tasks=[
      crocoit_chatbot_task  
    ],
    process=Process.sequential,
    
)

In [12]:
result = await  rankyx_crew.akickoff(
    inputs={
        "user_query": "what the Address of crocoit",
        "language": "English",
    }
)


print("Crew Result:", result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crocoit Technology Solutions Chatbot                                                                    │
│                                                                                                                 │
│  Task: Answer the user query: what the Address of crocoit                                                       │
│  If the query is about CrocoIT, use json_search_tool to retrieve information.                                   │
│  If the query is NOT about CrocoIT, answer from your own knowledge.                                             │
│  Do NOT use prior knowledge for CrocoIT-related questions.                                                      │
│  when query related to crocoit send sumry and direct answer not the source text                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool json_search_tool executed with result: [{'title': 'cotact_with_us -crocoit', 'content': 'cotact_with_us -crocoit\nCrocoIT is a technology solutions company based in Cairo, Egypt. Our office is located in Nasr City at 55 Sheikh Ahmed El-Saw...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crocoit Technology Solutions Chatbot                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "response": "CrocoIT is a technology solutions company based in Cairo, Egypt. The office is located in Nasr  │
│  City at 55 Sheikh Ahmed El-Sawy Street.",                                                                      │
│    "sources": [                                                                                                 │
│      {                                                                                                          │
│        "title": "cotact_with_us -crocoit",                                                                      │
│        "url": "https://crocoit.com/cotact_with_us/"                                                             │
│      }                                                                                                          │
│    ]                                                                                                            │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Crew Result: {'response': 'CrocoIT is a technology solutions company based in Cairo, Egypt. The office is located in Nasr City at 55 Sheikh Ahmed El-Sawy Street.', 'sources': [{'title': 'cotact_with_us -crocoit', 'url': 'https://crocoit.com/cotact_with_us/'}]}


In [1]:
print("ahmed")

ahmed
